In [ ]:
from pathlib import Path
import os

import numpy as np
import xarray as xr
from scipy.ndimage import gaussian_filter, convolve


# ============================================================
# Paths
# ============================================================

DATA_DIR = Path("../../data/processed/block_labeled_data")


# ============================================================
# Input and output variables
# ============================================================

BASE_VAR = "ssha_unfiltered"
QUALITY_VAR = "quality_flag"

GAUSSIAN_OUT_VAR = "gaussian_filter_highpass"
STEPPED_OUT_VAR = "stepped_convolution_highpass"

VALID_QUALITY_VALUES = (0, 5, 30, 50)


# ============================================================
# Filter parameters
# ============================================================

# SWOT pixel size used by the original filtering code.
PIXEL_SIZE_KM = 2.0

# Gaussian high-pass parameters.
GAUSSIAN_SIGMA = 8

# Stepped-convolution parameters.
INNER_RADIUS_KM = 5
MIDDLE_RADIUS_KM = 15
OUTER_RADIUS_KM = 30

INNER_WEIGHT = 5.0
MIDDLE_WEIGHT = 0.5
OUTER_WEIGHT = 0.2

INNER_RADIUS_PX = int(np.ceil(INNER_RADIUS_KM / PIXEL_SIZE_KM))
MIDDLE_RADIUS_PX = int(np.ceil(MIDDLE_RADIUS_KM / PIXEL_SIZE_KM))
OUTER_RADIUS_PX = int(np.ceil(OUTER_RADIUS_KM / PIXEL_SIZE_KM))


# ============================================================
# Quality masking
# ============================================================

def apply_quality_flag_mask(ssha, quality_flag):
    """
    Keep only pixels with quality_flag values 0, 5, 30, or 50.

    Values larger than 10 are also treated as invalid, matching the
    masking behavior from the original image-generation script.
    """
    ssha = np.asarray(ssha, dtype=np.float32)
    quality_flag = np.asarray(quality_flag)

    good_quality = np.isin(
        quality_flag,
        VALID_QUALITY_VALUES,
    )

    valid = (
        good_quality
        & np.isfinite(ssha)
        & (ssha <= 10)
    )

    return np.where(valid, ssha, np.nan).astype(np.float32)


# ============================================================
# NaN-aware filters
# ============================================================

def nan_gaussian_filter(arr, sigma):
    """
    Apply a Gaussian low-pass filter while ignoring NaN pixels.
    """
    arr = np.asarray(arr, dtype=np.float32)

    valid = np.isfinite(arr)

    arr_filled = np.where(
        valid,
        arr,
        0.0,
    ).astype(np.float32)

    weights = valid.astype(np.float32)

    smooth_sum = gaussian_filter(
        arr_filled,
        sigma=sigma,
        mode="nearest",
    )

    weight_sum = gaussian_filter(
        weights,
        sigma=sigma,
        mode="nearest",
    )

    smooth = smooth_sum / np.maximum(weight_sum, 1e-6)
    smooth[weight_sum <= 1e-6] = np.nan

    return smooth.astype(np.float32)


def gaussian_highpass(arr, sigma):
    """
    Gaussian high-pass:

        high-pass = original - Gaussian low-pass
    """
    arr = np.asarray(arr, dtype=np.float32)

    valid = np.isfinite(arr)

    smooth = nan_gaussian_filter(
        arr,
        sigma=sigma,
    )

    highpass = arr - smooth

    # Reapply the quality/original-data mask after filtering.
    highpass[~valid] = np.nan

    return highpass.astype(np.float32)


def build_stepped_kernel():
    """
    Build the normalized stepped radial convolution kernel.
    """
    kernel_radius_px = OUTER_RADIUS_PX

    y, x = np.ogrid[
        -kernel_radius_px:kernel_radius_px + 1,
        -kernel_radius_px:kernel_radius_px + 1,
    ]

    radius = np.sqrt(x**2 + y**2)

    kernel = np.zeros_like(
        radius,
        dtype=np.float32,
    )

    kernel[radius <= INNER_RADIUS_PX] = INNER_WEIGHT

    kernel[
        (radius > INNER_RADIUS_PX)
        & (radius <= MIDDLE_RADIUS_PX)
    ] = MIDDLE_WEIGHT

    kernel[
        (radius > MIDDLE_RADIUS_PX)
        & (radius <= OUTER_RADIUS_PX)
    ] = OUTER_WEIGHT

    kernel_sum = kernel.sum()

    if kernel_sum <= 0:
        raise ValueError(
            "The stepped-convolution kernel has a sum of zero."
        )

    kernel /= kernel_sum

    return kernel.astype(np.float32)


STEPPED_KERNEL = build_stepped_kernel()


def nan_stepped_convolution_filter(arr, kernel):
    """
    Apply the stepped-convolution low-pass filter while ignoring NaNs.
    """
    arr = np.asarray(arr, dtype=np.float32)

    valid = np.isfinite(arr)

    arr_filled = np.where(
        valid,
        arr,
        0.0,
    ).astype(np.float32)

    weights = valid.astype(np.float32)

    smooth_sum = convolve(
        arr_filled,
        weights=kernel,
        mode="nearest",
    )

    weight_sum = convolve(
        weights,
        weights=kernel,
        mode="nearest",
    )

    smooth = smooth_sum / np.maximum(weight_sum, 1e-6)
    smooth[weight_sum <= 1e-6] = np.nan

    return smooth.astype(np.float32)


def stepped_convolution_highpass(arr, kernel):
    """
    Stepped-convolution high-pass:

        high-pass = original - stepped-convolution low-pass
    """
    arr = np.asarray(arr, dtype=np.float32)

    valid = np.isfinite(arr)

    smooth = nan_stepped_convolution_filter(
        arr,
        kernel=kernel,
    )

    highpass = arr - smooth

    # Reapply the quality/original-data mask after filtering.
    highpass[~valid] = np.nan

    return highpass.astype(np.float32)


# ============================================================
# Apply filters to every 2D slice
# ============================================================

def process_all_slices(ssha, quality_flag):
    """
    Apply masking and both filters to an entire NetCDF variable.

    Supports:
        (height, width)
        (file, height, width)
        (..., height, width)

    The last two dimensions are treated as the spatial dimensions.
    """
    ssha = np.asarray(ssha, dtype=np.float32)
    quality_flag = np.asarray(quality_flag)

    if ssha.shape != quality_flag.shape:
        raise ValueError(
            "SSHA and quality-flag shapes do not match: "
            f"{ssha.shape} versus {quality_flag.shape}"
        )

    if ssha.ndim < 2:
        raise ValueError(
            f"Expected at least two dimensions, got {ssha.shape}"
        )

    gaussian_output = np.full(
        ssha.shape,
        np.nan,
        dtype=np.float32,
    )

    stepped_output = np.full(
        ssha.shape,
        np.nan,
        dtype=np.float32,
    )

    leading_shape = ssha.shape[:-2]

    if leading_shape:
        slice_indices = np.ndindex(leading_shape)
    else:
        slice_indices = [()]

    for index in slice_indices:
        ssha_slice = ssha[index]
        quality_slice = quality_flag[index]

        masked_ssha = apply_quality_flag_mask(
            ssha_slice,
            quality_slice,
        )

        gaussian_output[index] = gaussian_highpass(
            masked_ssha,
            sigma=GAUSSIAN_SIGMA,
        )

        stepped_output[index] = stepped_convolution_highpass(
            masked_ssha,
            kernel=STEPPED_KERNEL,
        )

    return gaussian_output, stepped_output


# ============================================================
# NetCDF processing
# ============================================================

processed_count = 0
skipped_count = 0
failed_items = []

nc_files = sorted(
    path
    for path in DATA_DIR.rglob("*.nc")
    if not path.name.endswith(".tmp.nc")
)

print(f"Looking in: {DATA_DIR.resolve()}")
print(f"NetCDF files found: {len(nc_files)}")

if not nc_files:
    raise FileNotFoundError(
        f"No NetCDF files were found in {DATA_DIR.resolve()}"
    )

print("\nFilter settings:")
print(
    f"Gaussian sigma: {GAUSSIAN_SIGMA} pixels "
    f"({GAUSSIAN_SIGMA * PIXEL_SIZE_KM} km)"
)

print("Stepped-convolution settings:")
print(
    f"  Inner:  {INNER_RADIUS_KM} km | "
    f"{INNER_RADIUS_PX} pixels | weight={INNER_WEIGHT}"
)
print(
    f"  Middle: {MIDDLE_RADIUS_KM} km | "
    f"{MIDDLE_RADIUS_PX} pixels | weight={MIDDLE_WEIGHT}"
)
print(
    f"  Outer:  {OUTER_RADIUS_KM} km | "
    f"{OUTER_RADIUS_PX} pixels | weight={OUTER_WEIGHT}"
)
print(f"  Kernel shape: {STEPPED_KERNEL.shape}")

for file_number, nc_file in enumerate(nc_files, start=1):
    print(
        f"\n[{file_number}/{len(nc_files)}] "
        f"Processing: {nc_file}"
    )

    temporary_file = nc_file.with_name(
        f"{nc_file.stem}.tmp.nc"
    )

    try:
        # Load the entire dataset into memory so that the original file
        # can be safely replaced after the open file handle is closed.
        with xr.open_dataset(nc_file) as source_ds:
            ds = source_ds.load()

        if BASE_VAR not in ds:
            skipped_count += 1
            failed_items.append(
                f"Missing {BASE_VAR}: {nc_file}"
            )
            print(f"Skipped: missing {BASE_VAR}")
            continue

        if QUALITY_VAR not in ds:
            skipped_count += 1
            failed_items.append(
                f"Missing {QUALITY_VAR}: {nc_file}"
            )
            print(f"Skipped: missing {QUALITY_VAR}")
            continue

        base_da = ds[BASE_VAR]
        quality_da = ds[QUALITY_VAR]

        if base_da.shape != quality_da.shape:
            skipped_count += 1
            failed_items.append(
                f"Shape mismatch: {nc_file} | "
                f"{BASE_VAR}={base_da.shape}, "
                f"{QUALITY_VAR}={quality_da.shape}"
            )
            print(
                "Skipped because variable shapes do not match: "
                f"{base_da.shape} versus {quality_da.shape}"
            )
            continue

        gaussian_values, stepped_values = process_all_slices(
            ssha=base_da.values,
            quality_flag=quality_da.values,
        )

        # Add or overwrite the Gaussian high-pass variable.
        ds[GAUSSIAN_OUT_VAR] = (
            base_da.dims,
            gaussian_values,
        )

        ds[GAUSSIAN_OUT_VAR].attrs = {
            "long_name": "Gaussian high-pass filtered SSHA",
            "description": (
                "Quality-masked SSHA minus a NaN-aware Gaussian "
                "low-pass filtered SSHA."
            ),
            "source_variable": BASE_VAR,
            "quality_mask_variable": QUALITY_VAR,
            "valid_quality_flag_values": "0, 5, 30, 50",
            "filter_type": "gaussian_highpass",
            "gaussian_sigma_pixels": GAUSSIAN_SIGMA,
            "gaussian_sigma_km": (
                GAUSSIAN_SIGMA * PIXEL_SIZE_KM
            ),
            "pixel_size_km": PIXEL_SIZE_KM,
        }

        # Add or overwrite the stepped-convolution variable.
        ds[STEPPED_OUT_VAR] = (
            base_da.dims,
            stepped_values,
        )

        ds[STEPPED_OUT_VAR].attrs = {
            "long_name": (
                "Stepped-convolution high-pass filtered SSHA"
            ),
            "description": (
                "Quality-masked SSHA minus a NaN-aware stepped "
                "radial convolution low-pass filtered SSHA."
            ),
            "source_variable": BASE_VAR,
            "quality_mask_variable": QUALITY_VAR,
            "valid_quality_flag_values": "0, 5, 30, 50",
            "filter_type": "stepped_convolution_highpass",
            "inner_radius_km": INNER_RADIUS_KM,
            "middle_radius_km": MIDDLE_RADIUS_KM,
            "outer_radius_km": OUTER_RADIUS_KM,
            "inner_radius_pixels": INNER_RADIUS_PX,
            "middle_radius_pixels": MIDDLE_RADIUS_PX,
            "outer_radius_pixels": OUTER_RADIUS_PX,
            "inner_weight": INNER_WEIGHT,
            "middle_weight": MIDDLE_WEIGHT,
            "outer_weight": OUTER_WEIGHT,
            "pixel_size_km": PIXEL_SIZE_KM,
        }

        # Copy the units from the original SSHA variable when available.
        if "units" in base_da.attrs:
            ds[GAUSSIAN_OUT_VAR].attrs["units"] = (
                base_da.attrs["units"]
            )
            ds[STEPPED_OUT_VAR].attrs["units"] = (
                base_da.attrs["units"]
            )

        # Store the new arrays as float32.
        ds[GAUSSIAN_OUT_VAR].encoding.update({
            "dtype": "float32",
            "zlib": True,
            "complevel": 4,
        })

        ds[STEPPED_OUT_VAR].encoding.update({
            "dtype": "float32",
            "zlib": True,
            "complevel": 4,
        })

        # Preserve unlimited dimensions when the original dataset has them.
        unlimited_dims = ds.encoding.get(
            "unlimited_dims",
            None,
        )

        # Write to a temporary file first. This prevents a partially
        # written file from replacing the original if writing fails.
        ds.to_netcdf(
            temporary_file,
            mode="w",
            unlimited_dims=unlimited_dims,
        )

        ds.close()

        # Atomically replace the original NetCDF file.
        os.replace(
            temporary_file,
            nc_file,
        )

        processed_count += 1

        print(f"Saved: {GAUSSIAN_OUT_VAR}")
        print(f"Saved: {STEPPED_OUT_VAR}")
        print(f"Updated file: {nc_file}")

    except Exception as error:
        if temporary_file.exists():
            temporary_file.unlink()

        skipped_count += 1
        failed_items.append(
            f"Failed: {nc_file} | {type(error).__name__}: {error}"
        )

        print(
            f"Failed: {type(error).__name__}: {error}"
        )


# ============================================================
# Final summary
# ============================================================

print("\nDone.")
print(f"NetCDF files checked: {len(nc_files)}")
print(f"NetCDF files updated: {processed_count}")
print(f"Skipped or failed: {skipped_count}")

if failed_items:
    print("\nSkipped/failed files:")
    for item in failed_items:
        print(f"  {item}")

In [1]:
from pathlib import Path
import shutil

import pandas as pd
import yaml


# ---------------------------------------------------------------------
# Paths
# ---------------------------------------------------------------------
csv_path = Path("../../data/splits/split_assignments.csv")
source_dir = Path("../../data/processed/block_labeled_data")
splits_dir = Path("../../data/splits")

yaml_path = splits_dir / "data.yaml"


# CSV split name -> output folder name
split_folder_map = {
    "train": "train",
    "val": "valid",
    "valid": "valid",
    "test": "test",
}


# ---------------------------------------------------------------------
# Create split directories
# ---------------------------------------------------------------------
for folder_name in {"train", "valid", "test"}:
    (splits_dir / folder_name).mkdir(parents=True, exist_ok=True)


# ---------------------------------------------------------------------
# Load CSV
# ---------------------------------------------------------------------
df = pd.read_csv(csv_path)

required_columns = {"image_name", "split"}
missing_columns = required_columns - set(df.columns)

if missing_columns:
    raise ValueError(
        f"CSV is missing required columns: {sorted(missing_columns)}"
    )


# ---------------------------------------------------------------------
# Find available NetCDF files
# ---------------------------------------------------------------------
nc_files = [
    file_path
    for file_path in source_dir.iterdir()
    if file_path.is_file() and file_path.suffix.lower() == ".nc"
]


copied_counts = {
    "train": 0,
    "valid": 0,
    "test": 0,
}

missing_files = []
multiple_matches = []


# ---------------------------------------------------------------------
# Copy each CSV entry into its assigned split folder
# ---------------------------------------------------------------------
for row in df.itertuples(index=False):
    image_name = str(row.image_name)
    csv_split = str(row.split).strip().lower()

    if csv_split not in split_folder_map:
        print(
            f"Skipping {image_name}: unsupported split '{csv_split}'"
        )
        continue

    output_split = split_folder_map[csv_split]

    # Example:
    # image:
    # SWOT_..._v3.0.jpg
    #
    # matching NetCDF:
    # SWOT_..._v3.0_labeled.nc
    image_prefix = Path(image_name).stem

    matches = [
        file_path
        for file_path in nc_files
        if file_path.name.startswith(image_prefix)
    ]

    if not matches:
        missing_files.append(image_name)
        continue

    if len(matches) > 1:
        multiple_matches.append(
            {
                "image_name": image_name,
                "matches": [file.name for file in matches],
            }
        )

    for source_file in matches:
        destination_file = splits_dir / output_split / source_file.name

        # copy2 preserves file metadata
        shutil.copy2(source_file, destination_file)

        copied_counts[output_split] += 1

        print(
            f"Copied: {source_file.name} -> "
            f"{output_split}/{source_file.name}"
        )


# ---------------------------------------------------------------------
# Create YOLO-style YAML file
# ---------------------------------------------------------------------
yaml_data = {
    "path": str(splits_dir.resolve()),
    "train": "train",
    "val": "valid",
    "test": "test",
    "names": {
        0: "iw",
    },
}

with yaml_path.open("w", encoding="utf-8") as yaml_file:
    yaml.safe_dump(
        yaml_data,
        yaml_file,
        sort_keys=False,
        default_flow_style=False,
    )


# ---------------------------------------------------------------------
# Summary
# ---------------------------------------------------------------------
print("\nCopy summary")
print("-" * 50)

for split_name, count in copied_counts.items():
    print(f"{split_name}: {count} files")

print(f"\nYAML created at: {yaml_path}")

if missing_files:
    print(f"\nMissing matching NetCDF files: {len(missing_files)}")

    for image_name in missing_files:
        print(f"  Missing: {image_name}")

if multiple_matches:
    print(f"\nCSV entries with multiple matches: {len(multiple_matches)}")

    for item in multiple_matches:
        print(f"  {item['image_name']}")

        for match in item["matches"]:
            print(f"    - {match}")

Copied: SWOT_L3_LR_SSH_Expert_001_161_20230726T224518_20230726T233644_v3.0_labeled.nc -> train/SWOT_L3_LR_SSH_Expert_001_161_20230726T224518_20230726T233644_v3.0_labeled.nc
Copied: SWOT_L3_LR_SSH_Expert_001_230_20230729T095509_20230729T104635_v3.0_labeled.nc -> train/SWOT_L3_LR_SSH_Expert_001_230_20230729T095509_20230729T104635_v3.0_labeled.nc
Copied: SWOT_L3_LR_SSH_Expert_001_245_20230729T224652_20230729T233818_v3.0_labeled.nc -> train/SWOT_L3_LR_SSH_Expert_001_245_20230729T224652_20230729T233818_v3.0_labeled.nc
Copied: SWOT_L3_LR_SSH_Expert_001_258_20230730T095540_20230730T104707_v3.0_labeled.nc -> valid/SWOT_L3_LR_SSH_Expert_001_258_20230730T095540_20230730T104707_v3.0_labeled.nc
Copied: SWOT_L3_LR_SSH_Expert_001_273_20230730T224723_20230730T233849_v3.0_labeled.nc -> train/SWOT_L3_LR_SSH_Expert_001_273_20230730T224723_20230730T233849_v3.0_labeled.nc
Copied: SWOT_L3_LR_SSH_Expert_001_286_20230731T095612_20230731T104738_v3.0_labeled.nc -> valid/SWOT_L3_LR_SSH_Expert_001_286_20230731T0